# 과제 2 — 병동별 낙상 발생률 (20점)

- **하는 일**: 가상 병원 CDW 로 병동별 낙상 발생률(재원 1,000일당)을 구합니다. 실제 병원 자료가 아닌 수업용 가상 자료입니다.
- **범위**: 12주차까지 배운 SQL (조회 · 집계 · JOIN · 서브쿼리 · 재원일수 · UPDATE · INSERT)
- **제출**: **2026-12-08(화)** 까지, LMS 에 `과제2.ipynb`

| 부분 | 문항 | 배점 | 채점 |
|---|---|---|---|
| 1. 단계별 SQL (8 · 9 · 12주차) | S1~S5 | 각 2점 (10점) | `___` 를 채워 실행해 ✅ 가 나오면 점수 |
| 2. 동료 SQL 점검 (12주차) | E1~E3 | 각 2점 (6점) | 틀린 한 군데를 고쳐 실행해 ✅ 가 나오면 점수 |
| 3. 검산표 (10주차) | T1 · T2 | 각 2점 (4점) | 위와 같음 |
| 합계 | 10문항 | 20점 | |

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.
- ✏️ 칸은 두 번 눌러 `→` 뒤에 적습니다.

### 푸는 방법

1. S · T 문항은 `___` 를 채워 실행하고, E 문항은 동료 SQL 의 한 군데를 고쳐 실행합니다.
2. 결과 아래에 ✅ 가 나오면 2점입니다. ❌ 가 두 번 나오면 💡 힌트가 보입니다.
3. 열 이름이나 값이 필요한 문항은 바로 위 조회 셀을 먼저 실행해 확인합니다.

In [ ]:
#@title ✏️ 학번 · 이름을 입력하고 실행하세요 (준비 셀) { display-mode: "form" }
학번 = ""  #@param {type:"string"}
이름 = ""  #@param {type:"string"}
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "hospital_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/hospital_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 1MB) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

학번, 이름 = str(globals().get("학번", "")).strip(), str(globals().get("이름", "")).strip()
if not 학번 or 학번 == "20260000":
    print("⚠️ 학번이 비어 있습니다. 위 칸에 학번과 이름을 입력하고 이 준비 셀을 다시 실행하세요.")
else:
    print(f"연결 완료.  {이름}({학번})")

def 제출확인():
    """제출 전에 실행: 문항별 결과와 제출 방법을 보여 줍니다."""
    print(f"제출자: {이름 or '(이름 없음)'} ({학번 or '학번 없음'})")
    진행확인()
    print("\n제출: ① Runtime → Run all (오류에서 멈추면 아래 셀을 누르고 Runtime → Run after)  ② File → Download → Download .ipynb  ③ LMS 에 과제2.ipynb 제출")

ANSWERS = _load_answers("eNqdVV1v2lYY/iuvuClMUYQDNvF22TJt0rZKTaZdTJOKP+giRc2UVNpFVYlQt6MhUWCFxmyASEtK2nHhDichE7vZT8mlz+v/sPfYBtlWEgkukPFzPnye533e5zxNrAmJz+FpYmfjET0Tkr6a1nKiqAhFNbEEia0b8O2tX3cIl+ivurXJ//6YYKM6O2zxUVZu4fNd55ONFTPxEwE/bzx+wrfxB0DgQx0b8I3N9mqA3Rf4Zw2Pami0ATsTDpeHbK+BgxI4Vom9H7rlPr0DHtk0mVUrrNqHJLbGrGvSUGo52APcehs7ZvD29T3AVgnYoIvnJ/6iZY/UtqZv6xodqFjY3NE58suTja3Hhc0p9IywtZWwMDlZEVRVK+Q0LRcWJo7fLgx2h5wpUYwJ41yMWbkH7HWfyKN5Cm7LxG4t6RPxX2hNipVNoJ9jGdMZ7m92ZAbpBdg0ON3OZAmmG1t98I9BMpM8FXZgBjsArWInE2AHDbf5YX6ZMhGZioqeVTPZrLwqRmSK4XP4Jy7bbJyd9KLmOjfYx9NlfxtwRhO/8m2gAY/nx38jBlqeDjh/WzSbhOEWC5b/dwHhUwCr17zF3LXdCu6dzS9UNiyUoMuCpKxI6UJGDgsVxwOhMmGhnIsJG9p4bN3aa2v5b/J318Grd28XOyV+/uRsbYodkzLDc+635gRmOBt4zeU22+C+ecX5kt/QXICvGOZblDKyosl6ThMLYb5x/HZjuIevnIvejV3k9wIfCLLE7x6iQj2Ag7Zn838OZuX3wd0JBQywywn7YPP44Q3Bu2hB3vlIoOqCJmpSUVTVTCRQ43jAWwjz9o9/TWnv3v/+u3VwbMM1x9TsFa/G/TobnWHHoDK/xGbP8/KgxgFmvFvEsPlIAKpKVlYKiiBrmhAmEsevI3KjRyUReE0y9CThuSOd0Se3SZ67NJyz2iJ5lI/kkZRbKabVtJxWZT1yn8XwuU6NbRL5RWmaG3TfONbvntAdg7fTa8sZW87lvnd9Uf7ulzgz3oQLVGE9YqdiprAqphVJ0FUl0kYxPOCzEuFTPXVGBh+9Ni28qPAu2ecV8GcksfIXBaRnpYaFpk3Xq2evjp1agh++yj/ITxc5ox5e9vzIoLT0PkU1tdl7A+5c1Y8h3LSQ9LM6dccrP3brFE+zL3g2sGpYHUc1g6t6DwJZmdVgpX08qmDbb9t+iSzDq8LOS65h0Qn6WKXDVvt0JLdpojHGPxrz6B51v0jXlyIVChkhEtdx/Lq4vlX3q/pb4Nn8MIhrr7WTn6Xgywf3vw2uoC8eAqd53GaH/NpvYmfsNns8pqJ3EdxcroXc9+x/qVKI8A==")   # 자동 확인용 (정답 SQL 은 없음)

## 데이터 구성 — 가상 병원 CDW

실제 병원 자료가 아니라 수업용으로 만든 가상 자료입니다 (2024년 · 병동 6곳). 열 이름이 생각나지 않으면 이 표로 돌아오세요.

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | 주로 쓰는 열 |
|---|---|---|---|---|
| `환자` | 환자 1명 | `환자ID` | — | 성별, 생년월일, 사망일 |
| `입원` | 입원 1건 | `입원ID` | `환자ID` → 환자, `병동` → 병동 | 병동, 진료과, 입원일, 퇴원일, 퇴원형태, 계획입원, 보험유형 |
| `진단` | 진단 1건 | `진단ID` | `입원ID` → 입원 | 진단코드(KCD), 주진단여부 |
| `검사` | 검사 1건 | `검사ID` | `환자ID` → 환자 | 검사일, 검사명, 결과값 |
| `병동` | 병동 1곳 | `병동` | — | 구분, 주진료과, 병상수 |
| `병동일보` | 병동 × 날짜 | `병동 + 날짜` | `병동` → 병동 | 재원환자수(자정 기준), 입원환자수, 퇴원환자수 |
| `근무` | 병동 × 날짜 × 근무조 | `병동 + 근무일 + 근무조` | `병동` → 병동 | 근무조(D · E · N), 간호사수 |
| `진료비` | 입원 × 항목 | `진료비ID` | `입원ID` → 입원 | 항목, 금액, 급여구분 |
| `낙상` | 낙상 1건 | `낙상ID` | `입원ID` → 입원 | 발생일, 근무조, 손상정도 |

- 환자 1명이 여러 번 입원하면 `입원` 표에 여러 행이 있습니다 (1 : N). 입원 1건에 진료비 · 진단 · 낙상도 여러 행일 수 있습니다.
- 표를 이을 때는 `ON 낙상.입원ID = 입원.입원ID` 처럼 외래키와 기본키를 맞춥니다.
- 2024-12-31 에 아직 재원 중인 입원은 `퇴원일` 이 비어 있습니다(NULL). 환자ID 는 앞자리 0 이 있는 글자입니다.

## 1. 단계별 SQL — 분자 · 분모 · 낙상률 (8 · 9 · 12주차)

낙상률(재원 1,000일당) = 낙상 건수 ÷ 재원일수 × 1,000. 분자와 분모를 따로 구한 뒤 잇습니다.

### S1 · 병동별 낙상 건수 (분자) (2점)

낙상 표에는 병동이 없어서 입원 표와 이어 병동을 붙입니다. 두 표를 잇는 **열 이름**을 채우세요.

💡 힌트: 바로 아래 셀에서 낙상 표의 열 이름을 봅니다. 두 표에 모두 있는 열로 잇습니다 (11강 「가상 병원 CDW — 핵심 네 표」).

In [ ]:
%%sql
-- 낙상 표의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 낙상);

In [ ]:
%%sql
-- S1 (2점)
-- ___ 를 채워 실행하세요
SELECT 입원.병동, COUNT(*) AS 낙상건수
FROM 낙상
JOIN 입원 ON 낙상.___ = 입원.입원ID
GROUP BY 입원.병동
ORDER BY 입원.병동;

### S2 · 병동별 재원일수 (분모) (2점)

분모는 재원일수입니다. 병동일보에서 ‘그날 자정의 재원 환자 수’를 1년치 더하면 재원일수가 됩니다 (12강 「병동일보」). 더할 **열 이름**을 채우세요.

💡 힌트: 바로 아래 셀에서 병동일보의 열과 값을 봅니다.

In [ ]:
%%sql
-- 병동일보의 열과 값 보기
SELECT *
FROM 병동일보
LIMIT 3;

In [ ]:
%%sql
-- S2 (2점)
-- ___ 를 채워 실행하세요
SELECT 병동, SUM(___) AS 재원일수
FROM 병동일보
GROUP BY 병동
ORDER BY 병동;

### S3 · 병동별 낙상률 (재원 1,000일당) (2점)

S1(분자)과 S2(분모)를 괄호 안에 넣어 병동으로 이었습니다. 낙상률 = 낙상건수 ÷ 재원일수 × 1,000. 두 결과를 잇는 **열 이름**을 채우세요.

💡 힌트: ON 에는 양쪽 결과에 모두 있는 열을 적습니다 (8강 JOIN · 9강 서브쿼리).

In [ ]:
%%sql
-- S3 (2점)
-- ___ 를 채워 실행하세요
SELECT 분모.병동, 분자.낙상건수, 분모.재원일수,
       ROUND(1000.0 * 분자.낙상건수 / 분모.재원일수, 2) AS 낙상률
FROM (SELECT 병동, SUM(재원환자수) AS 재원일수
      FROM 병동일보 GROUP BY 병동) AS 분모
JOIN (SELECT 입원.병동, COUNT(*) AS 낙상건수
      FROM 낙상 JOIN 입원 ON 낙상.입원ID = 입원.입원ID
      GROUP BY 입원.병동) AS 분자
  ON 분모.병동 = 분자.___
ORDER BY 분모.병동;

### S4 · 근무조별 낙상 건수 (2점)

근무조(D 낮 · E 초번 · N 밤)별로 낙상 건수를 셉니다. 묶을 **열 이름**을 채우세요.

💡 힌트: 바로 아래 셀에서 근무조 값을 봅니다 (8강 GROUP BY).

In [ ]:
%%sql
-- 근무조에 어떤 값이 있는지 보기
SELECT DISTINCT 근무조
FROM 낙상
ORDER BY 근무조;

In [ ]:
%%sql
-- S4 (2점)
-- ___ 를 채워 실행하세요
SELECT 근무조, COUNT(*) AS 낙상건수
FROM 낙상
GROUP BY ___
ORDER BY 근무조;

### S5 · 낙상이 있었던 입원의 평균 재원일수 (2점)

낙상이 있었던 입원만 골라(IN) 병동별 평균 재원일수를 구합니다. 재원일수 = 퇴원일 − 입원일 (12강). 빼는 **열 이름**을 채우세요. 재원 중인 입원(퇴원일 NULL)은 뺍니다.

💡 힌트: 바로 아래 셀에서 입원 표의 열 이름을 봅니다 (12강 「재원일수 — 퇴원일 − 입원일」 · 9강 서브쿼리 IN).

In [ ]:
%%sql
-- 입원 표의 열 이름 · 타입 보기
SELECT *
FROM (DESCRIBE 입원);

In [ ]:
%%sql
-- S5 (2점)
-- ___ 를 채워 실행하세요
SELECT 병동, ROUND(AVG(퇴원일 - ___), 1) AS 평균재원일수
FROM 입원
WHERE 퇴원일 IS NOT NULL
  AND 입원ID IN (SELECT 입원ID FROM 낙상)
GROUP BY 병동
ORDER BY 병동;

## 2. 동료 SQL 점검 (12주차)

동료의 SQL 3개는 **오류 없이 실행되지만 숫자가 틀립니다**. 틀린 한 줄을 찾아 고치세요.

### E1 · 낙상이 있었던 입원 건수 (2점)

동료가 원한 것: 낙상이 한 번이라도 있었던 입원은 몇 건인가요? (한 입원에서 두 번 넘어져도 1건)

💡 힌트: 한 입원에서 여러 번 넘어지면 낙상 표에 행이 여러 개입니다. 무엇을 세야 하나요? (8강 COUNT(DISTINCT 열) · 12강 「입원 건수와 환자 수」)

In [ ]:
%%sql
-- E1 (2점)
-- [고칠 SQL] 동료가 쓴 SQL — 오류 없이 실행되지만 숫자가 틀립니다. 한 군데를 고쳐 실행하세요
SELECT COUNT(입원ID) AS 입원건수
FROM 낙상;

### E2 · 65세 이상 환자의 낙상 건수 (2점)

동료가 원한 것: 65세 이상 환자의 낙상은 몇 건인가요? (날짜끼리 빼면 ‘일 수’가 나옵니다)

💡 힌트: 발생일 − 생년월일 은 일 수입니다. 65세를 일 수로 바꾸면? (11강 「요청서를 SQL 조건으로 옮기기」의 19 * 365)

In [ ]:
%%sql
-- E2 (2점)
-- [고칠 SQL] 동료가 쓴 SQL — 오류 없이 실행되지만 숫자가 틀립니다. 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 낙상건수
FROM 낙상
JOIN 입원 ON 낙상.입원ID = 입원.입원ID
JOIN 환자 ON 입원.환자ID = 환자.환자ID
WHERE 낙상.발생일 - 환자.생년월일 >= 65;

### E3 · 중등도 이상 손상 낙상 건수 (2점)

동료가 원한 것: 손상 정도가 ‘중등도 이상’인 낙상은 몇 건인가요?

💡 힌트: 바로 아래 셀에서 손상정도 값의 철자 · 띄어쓰기를 확인합니다 (12강 앞자리 0 처럼 글자 값은 똑같이 적어야 함).

In [ ]:
%%sql
-- 손상정도에 어떤 값이 있는지 보기
SELECT DISTINCT 손상정도
FROM 낙상
ORDER BY 손상정도;

In [ ]:
%%sql
-- E3 (2점)
-- [고칠 SQL] 동료가 쓴 SQL — 오류 없이 실행되지만 숫자가 틀립니다. 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 낙상건수
FROM 낙상
WHERE 손상정도 = '중등도이상';

## 3. 검산표 만들기 — UPDATE · INSERT (10주차)

검산표는 단계마다 건수를 적어 두는 표입니다 (9주차 검산). 내가 만든 연습용 표(TEMP)라 원본 데이터는 그대로입니다.

In [ ]:
%%sql
-- [검산표 준비] 실행만 하세요. 다시 할 때도 이 셀부터 실행합니다
CREATE OR REPLACE TEMP TABLE 검산표 (단계 VARCHAR, 건수 INTEGER);
INSERT INTO 검산표 VALUES ('① 입원 건수', 10030), ('② 재원일수 (분모)', 70343);

SELECT * FROM 검산표 ORDER BY 단계;

### T1 · 잘못 적은 건수 고치기 — UPDATE (2점)

동료가 검산표의 ‘② 재원일수 (분모)’ 를 70,343 로 잘못 적었습니다. **S2 결과를 모두 더한 값**(전체 재원일수)으로 고치세요. WHERE 를 빠뜨리면 모든 행이 바뀝니다.

💡 힌트: 10강 「UPDATE — 값 바꾸기」 · 「WHERE는 바꿀 대상을 정합니다」. 전체 재원일수는 `SELECT SUM(재원환자수) FROM 병동일보;` 로도 확인할 수 있습니다. 다시 할 때는 검산표 준비 셀부터 실행하세요.

In [ ]:
%%sql
-- T1 (2점)
-- ___ 를 채워 실행하세요
UPDATE 검산표
SET 건수 = ___
WHERE 단계 = ___;

-- 결과 확인 (그대로 두세요)
SELECT * FROM 검산표 WHERE 단계 IN ('① 입원 건수', '② 재원일수 (분모)') ORDER BY 단계;

### T2 · 나머지 단계 넣기 — INSERT (2점)

앞 문항에서 구한 건수를 검산표에 넣으세요. 단계 이름은 적혀 있고, 건수만 채웁니다: ③ = S1 낙상건수의 합(전체 낙상 건수) · ④ = S4 의 N · ⑤ = E2 를 고친 결과.

💡 힌트: 10강 「INSERT — 새 행 추가하기」. 같은 행이 두 번 들어갔으면 검산표 준비 셀부터 다시 실행하세요.

In [ ]:
%%sql
-- T2 (2점)
-- ___ 를 채워 실행하세요
INSERT INTO 검산표 VALUES
  ('③ 낙상 건수 (분자)', ___),
  ('④ 밤(N) 근무 낙상', ___),
  ('⑤ 65세 이상 낙상', ___);

-- 결과 확인 (그대로 두세요)
SELECT * FROM 검산표 WHERE 단계 NOT IN ('① 입원 건수', '② 재원일수 (분모)') ORDER BY 단계;

In [ ]:
%%sql
-- 검산표 전체 보기
SELECT * FROM 검산표 ORDER BY 단계;

In [ ]:
%%sql
-- 전체 낙상률(재원 1,000일당) = ③ ÷ ② × 1,000 — 검산표에서 꺼내 계산 (S3 의 병동별 낙상률과 비교해 보세요)
SELECT ROUND(1000.0 * (SELECT 건수 FROM 검산표 WHERE 단계 = '③ 낙상 건수 (분자)')
                    / (SELECT 건수 FROM 검산표 WHERE 단계 = '② 재원일수 (분모)'), 2) AS 낙상률;

## 제출 확인

In [ ]:
제출확인()

**제출** — 모든 문항이 ✅ 인 상태로 저장(`File → Download → Download .ipynb`)한 뒤 **LMS** 에 제출하세요 (기한 2026-12-08(화)).